In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm umap-learn

import os, json, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR = f"{OUT}/results"
FIGURES_DIR = f"{OUT}/figures"

classes = sorted([d for d in os.listdir(TEA_LEAF_ROOT)
                  if os.path.isdir(os.path.join(TEA_LEAF_ROOT, d))])
NUM_CLASSES = len(classes)
CLASS_TO_IDX = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]

# Load DINOv1
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Collect all images
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(TEA_LEAF_ROOT, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# Fixed split
random.seed(42)
shuffled = all_images.copy()
random.shuffle(shuffled)
n_test = int(0.2 * len(shuffled))
test_images = shuffled[:n_test]
train_images = shuffled[n_test:]

tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

print(f"✅ Ready: {len(all_images)} images, {NUM_CLASSES} classes")
print(f"   Train: {len(train_images)}, Test: {len(test_images)}")

In [ ]:
# EXPERIMENT: t-SNE and UMAP visualization of DINOv1 CLS features
# Shows that DINOv1 features naturally separate healthy from diseased

import umap

N_VIS = 200  # per class for visualization

print("="*60)
print("EXPERIMENT: Feature Clustering (t-SNE + UMAP)")
print("="*60)

# Extract CLS features
vis_features = []
vis_labels = []
vis_classes = []
vis_binary = []

print("Extracting DINOv1 CLS features...")

with torch.no_grad():
    for cls_name in tqdm(classes):
        cls_dir = os.path.join(TEA_LEAF_ROOT, cls_name)
        imgs = sorted([f for f in os.listdir(cls_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        step = max(1, len(imgs) // N_VIS)
        selected = imgs[::step][:N_VIS]

        for fname in selected:
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
            img_tensor = tf_dv1(img).unsqueeze(0).to(device)
            feat = dino_v1(img_tensor)  # CLS token feature [1, 384]
            vis_features.append(feat.cpu().numpy().flatten())
            vis_labels.append(CLASS_TO_IDX[cls_name])
            vis_classes.append(cls_name)
            vis_binary.append(0 if cls_name in healthy_classes else 1)

vis_features = np.stack(vis_features)
vis_labels = np.array(vis_labels)
vis_classes = np.array(vis_classes)
vis_binary = np.array(vis_binary)

print(f"Features: {vis_features.shape}")
torch.cuda.empty_cache()

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_2d = tsne.fit_transform(vis_features)

# UMAP
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_2d = reducer.fit_transform(vis_features)

# PCA (for completeness)
from sklearn.decomposition import PCA
print("Running PCA...")
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(vis_features)
pca_var = pca.explained_variance_ratio_

# ============================================================
# FIGURE: 2×3 grid (t-SNE, UMAP, PCA) × (by class, by binary)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Color maps
class_colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
binary_colors = {0: '#2ecc71', 1: '#e74c3c'}

embeddings = {
    't-SNE': tsne_2d,
    'UMAP': umap_2d,
    f'PCA ({pca_var[0]:.1%}+{pca_var[1]:.1%})': pca_2d,
}

for col, (name, emb) in enumerate(embeddings.items()):
    # Row 1: By disease class
    for cls_idx in range(NUM_CLASSES):
        mask = vis_labels == cls_idx
        axes[0, col].scatter(emb[mask, 0], emb[mask, 1],
                              c=[class_colors[cls_idx]], label=classes[cls_idx],
                              alpha=0.6, s=15, edgecolors='none')
    axes[0, col].set_title(f'{name}\n(by disease class)', fontsize=12, fontweight='bold')
    axes[0, col].legend(fontsize=7, markerscale=2, loc='best')
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    # Row 2: Binary (healthy vs diseased)
    for label, color in binary_colors.items():
        mask = vis_binary == label
        label_name = 'Healthy' if label == 0 else 'Diseased'
        axes[1, col].scatter(emb[mask, 0], emb[mask, 1],
                              c=color, label=label_name,
                              alpha=0.5, s=15, edgecolors='none')
    axes[1, col].set_title(f'{name}\n(healthy vs diseased)', fontsize=12, fontweight='bold')
    axes[1, col].legend(fontsize=10, markerscale=2)
    axes[1, col].set_xticks([]); axes[1, col].set_yticks([])

plt.suptitle('DINOv1 ViT-S/16 CLS Token Feature Clustering\n'
             'Zero-shot features naturally separate disease classes',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_tsne_umap_clustering.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(FIGURES_DIR, 'fig_tsne_umap_clustering.pdf'), bbox_inches='tight')
plt.show()
print("✅ Saved clustering figure")

In [ ]:
# FIXED: Copy dataset to local disk first, then train
# Drive random I/O is 10-50x slower than local disk

import shutil

LOCAL_DATA = "/tmp/tea_leaf_local"

if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (one-time, ~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
    print(f"✅ Copied to {LOCAL_DATA}")
else:
    print(f"✅ Local cache exists: {LOCAL_DATA}")

# Update paths to local
train_images_local = []
for s in train_images:
    s_local = s.copy()
    s_local['path'] = s['path'].replace(TEA_LEAF_ROOT, LOCAL_DATA)
    train_images_local.append(s_local)

test_images_local = []
for s in test_images:
    s_local = s.copy()
    s_local['path'] = s['path'].replace(TEA_LEAF_ROOT, LOCAL_DATA)
    test_images_local.append(s_local)

# Load pseudo-labels from saved results
pseudo_path = os.path.join(RESULTS_DIR, 'pipeline_pseudo_labels.json')
if os.path.exists(pseudo_path):
    print("Loading existing pseudo-labels...")
    # Regenerate pseudo-labels (fast, scores already computed pattern)
    # Use the saved UAHS heads
    uahs_heads = [1, 0]
else:
    uahs_heads = [1, 0]

# Quick pseudo-label generation using local disk
print("Scoring training images (local disk — much faster)...")

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn
def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads, C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = (cls_q @ k.transpose(-2, -1)) * scale
    cls_attn = cls_attn.softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = (q @ k.transpose(-2, -1)) * scale
    attn_full = attn_full.softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x
dv1_last_attn.forward = _dv1_patched_forward

DV1_GRID = 14
topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))

train_scores = []
with torch.no_grad():
    for s in tqdm(train_images_local, desc="Scoring"):
        try:
            img = Image.open(s['path']).convert('RGB')
            _ = dino_v1(tf_dv1(img).unsqueeze(0).to(device))
            attn = _dv1_store['cls_attn'][0].numpy()
            a = attn[uahs_heads].mean(axis=0)
            a_n = a / (a.sum() + 1e-10)
            train_scores.append(np.sort(a_n)[::-1][:topk].sum())
        except:
            train_scores.append(0.0)

train_scores = np.array(train_scores)
train_true_binary = np.array([s['is_diseased'] for s in train_images_local])
healthy_frac = (train_true_binary == 0).mean()
threshold = np.percentile(train_scores, healthy_frac * 100)
pseudo_labels = (train_scores > threshold).astype(int)
print(f"Pseudo-label accuracy: {accuracy_score(train_true_binary, pseudo_labels):.4f}")

torch.cuda.empty_cache()

# ---- Training with loss tracking (LOCAL DISK = FAST) ----
class SimpleDS(Dataset):
    def __init__(self, image_list, labels, transform):
        self.images = image_list
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, image_list, transform):
        self.images = image_list
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]['path']).convert('RGB')
        return self.transform(img), self.images[idx]['is_diseased']

test_ds = TestDS(test_images_local, tf_test)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
test_true = np.array([s['is_diseased'] for s in test_images_local])

def train_with_tracking(train_imgs, train_labels, epochs=30, label_name=""):
    ds = SimpleDS(train_imgs, train_labels, tf_train)
    loader = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    model = model.to(device)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'val_auc': [], 'val_acc': []}

    for ep in range(epochs):
        model.train()
        ep_loss, ep_correct, ep_total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device).long()
            out = model(imgs)
            loss = crit(out, labels)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item() * len(labels)
            ep_correct += (out.argmax(1) == labels).sum().item()
            ep_total += len(labels)
        sched.step()

        history['train_loss'].append(ep_loss / ep_total)
        history['train_acc'].append(ep_correct / ep_total)

        model.eval()
        probs, preds = [], []
        with torch.no_grad():
            for imgs, true_bin in test_loader:
                out = model(imgs.to(device))
                probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
                preds.append(out.argmax(1).cpu().numpy())
        probs = np.concatenate(probs)
        preds = np.concatenate(preds)

        history['val_auc'].append(float(roc_auc_score(test_true, probs)))
        history['val_acc'].append(float(accuracy_score(test_true, preds)))

        if (ep + 1) % 10 == 0:
            print(f"    {label_name} Epoch {ep+1}: loss={history['train_loss'][-1]:.4f}, "
                  f"val_auc={history['val_auc'][-1]:.4f}")

    del model; torch.cuda.empty_cache()
    return history

EPOCHS = 30

# 1. Pipeline (pseudo-labels)
print("\n--- Training with pseudo-labels ---")
hist_pseudo = train_with_tracking(train_images_local, pseudo_labels.tolist(), EPOCHS, "Pseudo")

# 2. Supervised (full real labels)
print("\n--- Training with real labels (full) ---")
hist_sup_full = train_with_tracking(train_images_local, train_true_binary.tolist(), EPOCHS, "SupFull")

# 3. Supervised (10/class = 70 labels)
print("\n--- Training with real labels (10/class) ---")
train_by_class = {}
for i, s in enumerate(train_images_local):
    train_by_class.setdefault(s['class_idx'], []).append(i)

random.seed(42)
sel_70 = []
for c in range(NUM_CLASSES):
    sel_70.extend(random.sample(train_by_class[c], min(10, len(train_by_class[c]))))
sel_imgs_70 = [train_images_local[i] for i in sel_70]
sel_labels_70 = [train_images_local[i]['is_diseased'] for i in sel_70]
hist_sup_70 = train_with_tracking(sel_imgs_70, sel_labels_70, 40, "Sup70")

# ---- FIGURE ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ep_x = list(range(1, EPOCHS + 1))
ep_x_70 = list(range(1, 41))

axes[0].plot(ep_x, hist_pseudo['train_loss'], '-', color='#e74c3c', lw=2, label='Pipeline (pseudo)')
axes[0].plot(ep_x, hist_sup_full['train_loss'], '-', color='#2563eb', lw=2, label='Supervised (full)')
axes[0].plot(ep_x_70, hist_sup_70['train_loss'], '--', color='#2563eb', lw=1.5, label='Supervised (70)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title('(a) Training Loss', fontweight='bold'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(ep_x, hist_pseudo['val_auc'], '-', color='#e74c3c', lw=2, label='Pipeline (pseudo)')
axes[1].plot(ep_x, hist_sup_full['val_auc'], '-', color='#2563eb', lw=2, label='Supervised (full)')
axes[1].plot(ep_x_70, hist_sup_70['val_auc'], '--', color='#2563eb', lw=1.5, label='Supervised (70)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation AUC')
axes[1].set_title('(b) Validation AUC', fontweight='bold'); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3); axes[1].set_ylim(0.5, 1.05)

axes[2].plot(ep_x, hist_pseudo['val_acc'], '-', color='#e74c3c', lw=2, label='Pipeline (pseudo)')
axes[2].plot(ep_x, hist_sup_full['val_acc'], '-', color='#2563eb', lw=2, label='Supervised (full)')
axes[2].plot(ep_x_70, hist_sup_70['val_acc'], '--', color='#2563eb', lw=1.5, label='Supervised (70)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Validation Accuracy')
axes[2].set_title('(c) Validation Accuracy', fontweight='bold'); axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3); axes[2].set_ylim(0.5, 1.05)

plt.suptitle('Training Dynamics: Pipeline (Pseudo-Labels) vs Supervised', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_training_loss_curves.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(FIGURES_DIR, 'fig_training_loss_curves.pdf'), bbox_inches='tight')
plt.show()

with open(os.path.join(RESULTS_DIR, 'training_loss_curves.json'), 'w') as f:
    json.dump({'pseudo': hist_pseudo, 'sup_full': hist_sup_full, 'sup_70': hist_sup_70}, f, indent=2)
print("✅ Saved loss curves")

In [ ]:
# EXPERIMENT: Run pipeline 3 times with different seeds
# Report mean ± std for all metrics

print("="*60)
print("EXPERIMENT: Multi-Run Pipeline (3 seeds)")
print("="*60)

# Map all images to the fast local disk cache
all_images_local = []
for s in all_images:
    s_local = s.copy()
    s_local['path'] = s['path'].replace(TEA_LEAF_ROOT, LOCAL_DATA)
    all_images_local.append(s_local)

# PRE-COMPUTE AND CACHE DINOv1 ATTENTION MAPS (ONE-TIME COST)
print("Pre-computing DINOv1 attention for all images to speed up seeds...")
attn_cache = {}
with torch.no_grad():
    for s in tqdm(all_images_local, desc="Caching DINOv1 Features"):
        try:
            if s['path'] not in attn_cache:
                img = Image.open(s['path']).convert('RGB')
                _ = dino_v1(tf_dv1(img).unsqueeze(0).to(device))
                attn_cache[s['path']] = _dv1_store['cls_attn'][0].numpy()
        except:
            pass

N_SEEDS = 3
multi_run_results = []

for seed in range(N_SEEDS):
    print(f"\n--- Seed {seed+1}/{N_SEEDS} ---")
    random.seed(seed * 1000 + 42)

    # Shuffle and split differently each time using LOCAL paths
    shuffled_s = all_images_local.copy()
    random.shuffle(shuffled_s)
    n_test_s = int(0.2 * len(shuffled_s))
    test_s = shuffled_s[:n_test_s]
    train_s = shuffled_s[n_test_s:]

    # UAHS head discovery
    pool = random.sample(train_s, min(200, len(train_s)))
    head_ent = {h: [] for h in range(6)}
    with torch.no_grad():
        for sample in pool:
            if sample['path'] in attn_cache:
                attn = attn_cache[sample['path']]
                for h in range(6):
                    a = attn[h]; a = a/(a.sum()+1e-10)
                    head_ent[h].append(-(a*np.log(a+1e-10)).sum())
    sorted_heads = sorted(range(6), key=lambda h: -np.std(head_ent[h]))
    sel_heads = sorted_heads[:2]
    print(f"  UAHS heads: {sel_heads}")

    # Score training images using CACHE
    scores = []
    for s in train_s:
        if s['path'] in attn_cache:
            attn = attn_cache[s['path']]
            a = attn[sel_heads].mean(axis=0)
            a_n = a / (a.sum() + 1e-10)
            scores.append(np.sort(a_n)[::-1][:topk].sum())
        else:
            scores.append(0.0)

    scores = np.array(scores)
    true_bin = np.array([s['is_diseased'] for s in train_s])

    # Pseudo-labels
    h_frac = (true_bin == 0).mean()
    thresh = np.percentile(scores, h_frac * 100)
    pseudo = (scores > thresh).astype(int)
    p_acc = accuracy_score(true_bin, pseudo)
    p_f1 = f1_score(true_bin, pseudo)
    print(f"  Pseudo acc: {p_acc:.4f}, F1: {p_f1:.4f}")

    # Train MobileNetV3
    ds_t = SimpleDS(train_s, pseudo.tolist(), tf_train)
    ds_e = TestDS(test_s, tf_test)
    loader_t = DataLoader(ds_t, batch_size=64, shuffle=True, num_workers=2)
    loader_e = DataLoader(ds_e, batch_size=64, shuffle=False, num_workers=2)
    test_true_s = np.array([s['is_diseased'] for s in test_s])

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
    crit = nn.CrossEntropyLoss()

    model.train()
    for ep in range(30):
        for imgs, labels in loader_t:
            imgs, labels = imgs.to(device), labels.to(device).long()
            loss = crit(model(imgs), labels)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

    model.eval()
    probs_all, preds_all = [], []
    with torch.no_grad():
        for imgs, true_bin_t in loader_e:
            out = model(imgs.to(device))
            probs_all.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
            preds_all.append(out.argmax(1).cpu().numpy())
    probs_all = np.concatenate(probs_all)
    preds_all = np.concatenate(preds_all)

    auc = roc_auc_score(test_true_s, probs_all)
    acc = accuracy_score(test_true_s, preds_all)
    f1 = f1_score(test_true_s, preds_all)

    print(f"  Pipeline: AUC={auc:.4f}, Acc={acc:.4f}, F1={f1:.4f}")

    multi_run_results.append({
        'seed': seed,
        'uahs_heads': sel_heads,
        'pseudo_acc': float(p_acc),
        'pseudo_f1': float(p_f1),
        'pipeline_auc': float(auc),
        'pipeline_acc': float(acc),
        'pipeline_f1': float(f1),
    })

    del model; torch.cuda.empty_cache()

# Summary
print(f"\n{'='*60}")
print("MULTI-RUN SUMMARY (mean ± std)")
print(f"{'='*60}")

aucs = [r['pipeline_auc'] for r in multi_run_results]
accs = [r['pipeline_acc'] for r in multi_run_results]
f1s = [r['pipeline_f1'] for r in multi_run_results]
paccs = [r['pseudo_acc'] for r in multi_run_results]

print(f"  Pipeline AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Pipeline Acc:     {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"  Pipeline F1:      {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  Pseudo-label Acc: {np.mean(paccs):.4f} ± {np.std(paccs):.4f}")

# Save
with open(os.path.join(RESULTS_DIR, 'multi_run_pipeline.json'), 'w') as f:
    json.dump({
        'runs': multi_run_results,
        'summary': {
            'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
            'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
            'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
            'pseudo_acc_mean': float(np.mean(paccs)), 'pseudo_acc_std': float(np.std(paccs)),
        }
    }, f, indent=2)
print("✅ Saved multi-run results")

In [ ]:
# Generate formal hyperparameter table (save as JSON for paper)

print("="*60)
print("Hyperparameter Configuration")
print("="*60)

hyperparams = {
    'DINOv1_Feature_Extraction': {
        'Model': 'DINOv1 ViT-S/16',
        'Input_Resolution': '224×224',
        'Patch_Size': 16,
        'Grid_Size': '14×14 (196 patches)',
        'Feature_Dim': 384,
        'Attention_Heads': 6,
        'Pretrained': 'DINO self-distillation (ImageNet-1K)',
        'Frozen': True,
    },
    'UAHS_Head_Selection': {
        'Selection_Criterion': 'Entropy variance (std of per-image entropy)',
        'Pool_Size': 200,
        'Auto_K_Method': 'Marginal AUC gain > 0.002',
        'Selected_K': 2,
        'Typical_Heads': 'Head 0 + Head 1 (highest entropy variance)',
    },
    'Pseudo_Label_Generation': {
        'Scoring_Method': 'Attention concentration (top-10% patch mass)',
        'Thresholding': 'Percentile (healthy class fraction)',
        'Alternative': 'GMM (2-component Gaussian Mixture Model)',
        'Selection': 'Better accuracy of the two methods',
    },
    'MobileNetV3_Training': {
        'Architecture': 'MobileNetV3-Small',
        'Pretrained_Init': 'ImageNet-1K',
        'Classifier_Head': 'Linear(576→2)',
        'Optimizer': 'Adam',
        'Learning_Rate': 1e-3,
        'Scheduler': 'CosineAnnealing (T_max=30)',
        'Epochs': 30,
        'Batch_Size': 64,
        'Loss': 'CrossEntropyLoss',
        'Augmentation': 'HFlip + VFlip + Rotation(20°) + ColorJitter(0.2)',
    },
    'Supervised_Baselines': {
        'Models': ['MobileNetV3-Small', 'EfficientNet-B0', 'ResNet-18'],
        'Budgets_Per_Class': [5, 10, 25, 50, 'full'],
        'Seeds': 3,
        'Epochs_Small': '50 (≤25/class), 30 (>25/class)',
        'Same_Optimizer': True,
    },
}

# Print formatted
for section, params in hyperparams.items():
    print(f"\n  {section}:")
    for key, val in params.items():
        print(f"    {key}: {val}")

# Save
with open(os.path.join(RESULTS_DIR, 'hyperparameters.json'), 'w') as f:
    json.dump(hyperparams, f, indent=2, default=str)
print("\n✅ Saved hyperparameters")